# HFA AI Training - Module 3: Vector Databases
## Real Estate Semantic Search Demo

This notebook demonstrates a complete semantic search system using real estate listings as the dataset. It shows:

1. Loading a set of property listings
2. Chunking longer listing descriptions
3. Embedding with OpenAI (text-embedding-3-small) and storing them in Pinecone
4. Performing semantic search with natural language
5. Showing how queries match listings that don't share exact words

This is designed to be relatable for real estate professionals -- it shows how AI-powered search understands buyer intent even when the buyer's language doesn't match the listing's language.

### Install Dependencies

In [ ]:
!pip install pinecone openai

### Section 1: Property Listing Data

These are detailed property listings written in varied language. Some describe the same features using very different words. This is exactly the challenge in real estate -- every agent writes listings differently, but buyers search consistently.

In [ ]:
import os
import textwrap
import time
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI

# ============================================================
# SECTION 1: Our Property Listing Data
# ============================================================
# These are detailed property listings written in varied language.
# Some describe the same features using very different words.
# This is exactly the challenge in real estate — every agent
# writes listings differently, but buyers search consistently.

PROPERTY_LISTINGS = [
    {
        "id": "MLS-2024-001",
        "title": "Kailua Beachside Gem",
        "description": (
            "Nestled just steps from the powdery white sands of Kailua Beach, this "
            "meticulously maintained 3-bedroom, 2-bathroom residence offers the ultimate "
            "Hawaiian lifestyle. Wake up to the sound of waves and enjoy your morning "
            "coffee on the covered lanai with unobstructed views of the Mokulua Islands. "
            "The open-concept living area features vaulted ceilings, bamboo flooring, and "
            "large sliding glass doors that blur the line between indoor and outdoor living. "
            "Recently updated kitchen with quartz countertops and stainless steel appliances. "
            "Mature tropical landscaping provides privacy while maintaining that coveted "
            "beach cottage feel. Walking distance to Kailua Town shops and restaurants."
        ),
        "price": 1850000,
        "beds": 3,
        "baths": 2,
        "area": "Kailua",
        "island": "Oahu",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-002",
        "title": "Mililani Family Paradise",
        "description": (
            "Welcome to your dream family home in the heart of Mililani! This spacious "
            "4-bedroom, 3-bathroom two-story home sits on a quiet cul-de-sac surrounded "
            "by well-manicured gardens. The upgraded kitchen opens to a generous family "
            "room perfect for movie nights and gatherings. Upstairs, the primary suite "
            "features a walk-in closet and en-suite bathroom with dual vanities. Three "
            "additional bedrooms provide plenty of space for children or a home office. "
            "The large, fenced backyard is ideal for barbecues, a play structure, or even "
            "a future pool. Zoned for top-rated Mililani schools. Community amenities "
            "include pools, tennis courts, parks, and miles of walking trails. Minutes "
            "from shopping, dining, and the H-2 freeway for easy commuting."
        ),
        "price": 985000,
        "beds": 4,
        "baths": 3,
        "area": "Mililani",
        "island": "Oahu",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-003",
        "title": "Waikiki Luxury Tower Residence",
        "description": (
            "Experience resort-style living in this stunning 2-bedroom, 2-bathroom corner "
            "unit at one of Waikiki's premier high-rise towers. Floor-to-ceiling windows "
            "frame breathtaking panoramic views of Diamond Head, the Pacific Ocean, and "
            "the Honolulu city lights. The chef's kitchen boasts Italian marble countertops, "
            "Sub-Zero refrigerator, and Wolf range. Rich hardwood floors flow throughout "
            "the open living and dining areas. The building offers world-class amenities: "
            "infinity pool, fitness center, spa, 24-hour concierge, valet parking, and a "
            "private theater. Steps from world-famous Waikiki Beach, high-end shopping at "
            "the Royal Hawaiian Center, and Honolulu's finest dining establishments."
        ),
        "price": 2400000,
        "beds": 2,
        "baths": 2,
        "area": "Waikiki",
        "island": "Oahu",
        "type": "Condo",
    },
    {
        "id": "MLS-2024-004",
        "title": "North Shore Surfer's Dream",
        "description": (
            "Quintessential North Shore living in this charming 2-bedroom, 1-bathroom "
            "plantation-style cottage in Haleiwa. Perfectly positioned between Pipeline "
            "and Sunset Beach, this is a surfer's paradise. Original hardwood floors, "
            "high ceilings, and louver windows invite the trade winds through every room. "
            "The spacious covered porch is perfect for board storage, post-surf hangouts, "
            "or simply watching the legendary North Shore sunsets. Outdoor shower to rinse "
            "off the salt water. The lush half-acre lot includes mature fruit trees — "
            "mango, papaya, and breadfruit. Just a short bike ride to historic Haleiwa "
            "Town with its famous shrimp trucks, boutiques, and art galleries."
        ),
        "price": 895000,
        "beds": 2,
        "baths": 1,
        "area": "Haleiwa",
        "island": "Oahu",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-005",
        "title": "Ewa Beach Starter Home",
        "description": (
            "Great opportunity for first-time buyers! This well-maintained 3-bedroom, "
            "2-bathroom single-level home in Ocean Pointe offers an accessible entry into "
            "homeownership. The functional floor plan includes a bright, open kitchen with "
            "breakfast bar, a comfortable living room, and a private primary bedroom with "
            "en-suite bath. Low-maintenance yard with covered patio for weekend grilling. "
            "The Ocean Pointe community features parks, a recreation center, and family "
            "events throughout the year. Close to Ewa Beach shopping center and new "
            "developments bringing even more amenities to the area. Easy freeway access "
            "for commuting to town. VA and FHA financing welcome."
        ),
        "price": 625000,
        "beds": 3,
        "baths": 2,
        "area": "Ewa Beach",
        "island": "Oahu",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-006",
        "title": "Manoa Valley Tropical Retreat",
        "description": (
            "Escape to your own private paradise in verdant Manoa Valley. This unique "
            "2-bedroom, 1-bathroom home is surrounded by towering monkeypod trees and "
            "tropical flora on a secluded half-acre lot. The post-and-pier construction "
            "allows natural airflow, while the wraparound deck offers peaceful views of "
            "the Ko'olau Mountains. Inside, you'll find an artist's studio, a library "
            "nook, and a sun-drenched meditation space. The property backs up to Manoa "
            "Falls trail — perfect for morning hikes before the tourists arrive. Listen "
            "to native birds and the gentle sound of Manoa Stream from your bedroom. "
            "A true sanctuary for those seeking serenity over spectacle."
        ),
        "price": 1100000,
        "beds": 2,
        "baths": 1,
        "area": "Manoa",
        "island": "Oahu",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-007",
        "title": "Kakaako Urban Living",
        "description": (
            "Sleek and modern 1-bedroom, 1-bathroom unit in the heart of Kakaako's "
            "vibrant urban district. This 650-square-foot smart home features motorized "
            "blinds, integrated sound system, and keyless entry. The galley kitchen has "
            "premium European cabinetry and stone countertops. In-unit washer/dryer and "
            "split AC for comfort. One secured parking stall with EV charging capability. "
            "The building's rooftop offers a BBQ area, lap pool, and panoramic views. "
            "You're walking distance to SALT at Our Kaka'ako, Ward Village shops, "
            "Ala Moana Center, and some of Honolulu's best restaurants and nightlife. "
            "Perfect for the young professional seeking a lock-and-leave lifestyle."
        ),
        "price": 545000,
        "beds": 1,
        "baths": 1,
        "area": "Kakaako",
        "island": "Oahu",
        "type": "Condo",
    },
    {
        "id": "MLS-2024-008",
        "title": "Big Island Coffee Farm Estate",
        "description": (
            "Live the agricultural dream on this 5-acre working Kona coffee farm. The "
            "3-bedroom, 2-bathroom main residence features a commercial-grade kitchen, "
            "wraparound lanai with coastline views, and a detached guest cottage for "
            "visitors or rental income. Over 1,000 coffee trees produce premium Kona "
            "coffee beans, with an established processing facility on-site. The property "
            "also includes macadamia nut trees, a vegetable garden, and chicken coop. "
            "Located in the famous Kona Coffee Belt at 1,500 feet elevation, you'll "
            "enjoy cooler temperatures, ample rainfall, and spectacular sunsets over "
            "the Pacific. A rare opportunity to own a piece of Hawaii's agricultural "
            "heritage while generating income."
        ),
        "price": 1650000,
        "beds": 3,
        "baths": 2,
        "area": "Kona",
        "island": "Big Island",
        "type": "Farm/Estate",
    },
    {
        "id": "MLS-2024-009",
        "title": "Maui Oceanfront Retreat",
        "description": (
            "Wake up to whales breaching right from your bedroom in this exquisite "
            "3-bedroom, 3-bathroom oceanfront villa in Wailea. The great room's "
            "retractable glass walls open completely to a sprawling oceanside terrace "
            "with heated infinity pool and built-in spa. Gourmet kitchen with dual "
            "islands, wine storage, and butler's pantry for effortless entertaining. "
            "The primary suite occupies the entire second floor with private balcony, "
            "fireplace, and spa-inspired bathroom featuring a soaking tub overlooking "
            "the sea. Smart home technology throughout. Impeccably landscaped grounds "
            "with direct access to a protected snorkeling cove. Five-star resort living "
            "in the privacy of your own home."
        ),
        "price": 5800000,
        "beds": 3,
        "baths": 3,
        "area": "Wailea",
        "island": "Maui",
        "type": "Single Family",
    },
    {
        "id": "MLS-2024-010",
        "title": "Puna Off-Grid Homestead",
        "description": (
            "Embrace sustainable island living on this 3-acre parcel in lower Puna. "
            "The custom-built tiny home features a sleeping loft, composting toilet, "
            "and a surprisingly functional kitchenette. Completely off-grid with a "
            "robust 5kW solar panel system, battery backup, and rainwater catchment "
            "with UV filtration. The property includes an established permaculture "
            "food forest with banana, papaya, citrus, and breadfruit trees. A separate "
            "outdoor kitchen with a rocket stove and wood-fired pizza oven makes "
            "entertaining a joy. Located in a friendly, like-minded community of "
            "sustainability enthusiasts. Hear the coqui frogs at night and wake to "
            "birdsong. The ultimate low-footprint Hawaiian lifestyle."
        ),
        "price": 189000,
        "beds": 1,
        "baths": 1,
        "area": "Puna",
        "island": "Big Island",
        "type": "Tiny Home",
    },
]

### Section 2: Document Chunking

Long documents should be broken into smaller chunks before embedding. This way, when someone searches for a specific detail, the relevant chunk (not the entire document) is found.

For our listings, we'll chunk by splitting the description into overlapping segments. The overlap ensures that if a sentence is split across two chunks, the second chunk still has context from the end of the first.

In [ ]:
# ============================================================
# SECTION 2: Document Chunking
# ============================================================
# Long documents should be broken into smaller chunks before
# embedding. This way, when someone searches for a specific
# detail, the relevant chunk (not the entire document) is found.
#
# For our listings, we'll chunk by splitting the description
# into overlapping segments. For short listings like these,
# we might keep them whole, but we'll demonstrate chunking
# to show the concept.

def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping chunks.

    Args:
        text:       The text to split
        chunk_size: Maximum characters per chunk
        overlap:    Number of characters to overlap between chunks

    Returns:
        List of text chunks

    Why overlap? So that if a sentence is split across two chunks,
    the second chunk still has context from the end of the first.
    """
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        # Find the end of this chunk
        end = start + chunk_size

        # If we're not at the end of the text, try to break at a sentence
        if end < text_length:
            # Look for the last period or sentence break within the chunk
            last_period = text.rfind(". ", start, end)
            if last_period > start:
                end = last_period + 1  # Include the period

        chunk = text[start:end].strip()
        if chunk:  # Don't add empty chunks
            chunks.append(chunk)

        # Move the start forward, minus the overlap
        start = end - overlap if end < text_length else text_length

    return chunks

### Section 3: Process and Store Listings in Pinecone

We'll initialize Pinecone and OpenAI clients, chunk each listing description, generate embeddings with OpenAI (text-embedding-3-small), and store all chunks with metadata for filtering.

**Requirements:** Set `PINECONE_API_KEY` and `OPENAI_API_KEY` environment variables.

In [ ]:
# ============================================================
# SECTION 3: Process and Store Listings
# ============================================================

print("=" * 60)
print("Hawaii Real Estate Semantic Search")
print("=" * 60)

# Initialize Pinecone and OpenAI clients
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
openai_client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generate embeddings using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

# Create a Pinecone index for our listings
INDEX_NAME = "hawaii-real-estate"

if INDEX_NAME in [idx.name for idx in pc.list_indexes()]:
    pc.delete_index(INDEX_NAME)

pc.create_index(
    name=INDEX_NAME,
    dimension=EMBEDDING_DIM,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

time.sleep(1)  # Wait for index to be ready

index = pc.Index(INDEX_NAME)

print("\nProcessing listings...\n")

# Process each listing: chunk the description and store each chunk
all_ids = []
all_documents = []
all_metadatas = []

for listing in PROPERTY_LISTINGS:
    # Chunk the description
    chunks = chunk_text(listing["description"], chunk_size=300, overlap=50)

    for i, chunk in enumerate(chunks):
        chunk_id = f"{listing['id']}_chunk_{i}"

        # Prepend the title for context (helps the embedding capture
        # what property this chunk is about)
        document_text = f"{listing['title']}: {chunk}"

        # Store metadata so we can filter and display results
        metadata = {
            "mls_id": listing["id"],
            "title": listing["title"],
            "price": listing["price"],
            "beds": listing["beds"],
            "baths": listing["baths"],
            "area": listing["area"],
            "island": listing["island"],
            "type": listing["type"],
            "chunk_index": i,
            "total_chunks": len(chunks),
        }

        all_ids.append(chunk_id)
        all_documents.append(document_text)
        all_metadatas.append(metadata)

    print(f"  {listing['id']} | {listing['title']:<35} | {len(chunks)} chunks")

# Generate embeddings for all documents
print(f"\nGenerating embeddings for {len(all_documents)} chunks...")
embeddings = get_embeddings(all_documents)

# Build vectors with metadata (include text in metadata)
vectors = []
for doc_id, embedding, text, metadata in zip(all_ids, embeddings, all_documents, all_metadatas):
    meta = {**metadata, "text": text}
    vectors.append({"id": doc_id, "values": embedding, "metadata": meta})

# Upsert in batches of 100
batch_size = 100
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i:i+batch_size])

print(f"\nStored {len(all_ids)} total chunks from {len(PROPERTY_LISTINGS)} listings.")
print(f"Index '{INDEX_NAME}' ready for searching!")

### Section 4: Search Function

This helper function performs semantic search, deduplicates results (since one listing may have multiple matching chunks), and displays formatted results.

In [ ]:
# ============================================================
# SECTION 4: Semantic Search Demonstrations
# ============================================================
# Now the fun part! We'll run several natural language queries
# and show how semantic search finds relevant listings even
# when the query words DON'T appear in the listings.

def search_listings(query: str, n_results: int = 3, where_filter: dict = None):
    """
    Search listings by semantic similarity.

    Performs the search, deduplicates results (since one listing
    may have multiple matching chunks), and displays formatted results.
    """
    print(f"\n{'\u2500' * 60}")
    print(f"  SEARCH: \"{query}\"")
    if where_filter:
        print(f"  FILTER: {where_filter}")
    print(f"{'\u2500' * 60}\n")

    # Generate embedding for the query using OpenAI
    query_embedding = get_embeddings([query])[0]

    # Query Pinecone — find similar chunks by vector similarity
    query_params = {
        "vector": query_embedding,
        "top_k": n_results * 2,  # Get extra results before deduplication
        "include_metadata": True,
    }
    if where_filter:
        query_params["filter"] = where_filter

    results = index.query(**query_params)

    # Deduplicate: multiple chunks from the same listing may match.
    # Keep only the best-matching chunk per listing.
    seen_listings = set()
    unique_results = []

    for match in results.matches:
        mls_id = match.metadata["mls_id"]
        if mls_id not in seen_listings and len(unique_results) < n_results:
            seen_listings.add(mls_id)
            unique_results.append({
                "id": match.id,
                "document": match.metadata["text"],
                "metadata": {k: v for k, v in match.metadata.items() if k != "text"},
                "score": match.score,  # cosine similarity (higher = better)
            })

    # Display results
    for rank, result in enumerate(unique_results, 1):
        meta = result["metadata"]
        score = result["score"]

        # Cosine similarity is already 0-1 for normalized vectors (higher = better)
        similarity_pct = score * 100

        print(f"  #{rank}  {meta['title']}")
        print(f"      MLS: {meta['mls_id']} | {meta['area']}, {meta['island']}")
        print(f"      {meta['beds']}BR/{meta['baths']}BA {meta['type']} | ${meta['price']:,}")
        print(f"      Match strength: {similarity_pct:.1f}% (score: {score:.4f})")

        # Show a snippet of the matching chunk
        snippet = result["document"][:150]
        wrapped = textwrap.fill(snippet + "...", width=55, initial_indent="      ", subsequent_indent="      ")
        print(wrapped)
        print()

    return unique_results

### Demo 1: Different Words, Same Meaning

This is the key demonstration. The query uses words that **DO NOT** appear in any listing. Traditional keyword search would return ZERO results. Semantic search understands the MEANING and finds matches.

In [ ]:
# ============================================================
# DEMO 1: The Core Value — Different Words, Same Meaning
# ============================================================
# This is the key demonstration. The query uses words that
# DON'T appear in the listings, but semantic search still
# finds the right matches.

print("\n" + "=" * 60)
print("DEMO 1: Different Words, Same Meaning")
print("=" * 60)
print("""
  The following query uses words that do NOT appear in any listing.
  Traditional keyword search would return ZERO results.
  Semantic search understands the MEANING and finds matches.
""")

search_listings("affordable oceanfront property for a young couple starting out")
# Expected: Should find Ewa Beach starter home and possibly Kakaako condo
# Note: "affordable," "oceanfront," and "young couple" don't appear
# in the listings verbatim, but the meaning matches!

### Demo 2: Lifestyle-Based Search

Buyers often describe the LIFE they want, not the HOUSE they want. Semantic search bridges that gap.

In [ ]:
# ============================================================
# DEMO 2: Lifestyle-Based Search
# ============================================================
# Buyers often search by lifestyle, not property features.

print("\n" + "=" * 60)
print("DEMO 2: Lifestyle-Based Search")
print("=" * 60)
print("""
  Buyers describe the LIFE they want, not the HOUSE they want.
  Semantic search bridges that gap.
""")

search_listings("laid back island life where I can catch waves every morning")
# Expected: Should find the North Shore cottage and possibly Maui

In [ ]:
search_listings("peaceful retirement surrounded by nature far from the city")
# Expected: Should find Manoa Valley and possibly the Big Island properties

### Demo 3: Semantic Search + Filters

In practice, you combine meaning-based search with hard filters. This gives you the best of both worlds -- natural language understanding with structured constraints.

In [ ]:
# ============================================================
# DEMO 3: Combining Semantic Search with Filters
# ============================================================
# In practice, you combine meaning-based search with hard filters.

print("\n" + "=" * 60)
print("DEMO 3: Semantic Search + Filters")
print("=" * 60)
print("""
  Combine natural language search with structured filters
  for the best of both worlds.
""")

search_listings(
    "entertaining guests with beautiful views",
    n_results=3,
    where_filter={"beds": {"$gte": 3}},
)
# Expected: Should find properties with 3+ beds that mention
# views and entertaining, like Wailea or Kailua

In [ ]:
search_listings(
    "investment property that generates income",
    n_results=3,
    where_filter={"island": {"$eq": "Big Island"}},
)
# Expected: Should find the Kona coffee farm (income-generating)

### Demo 4: Specific Feature Search

Searching for specific features -- even when described with completely different terminology.

In [ ]:
# ============================================================
# DEMO 4: Specific Feature Search
# ============================================================
# Searching for specific features described in different ways.

print("\n" + "=" * 60)
print("DEMO 4: Specific Feature Search")
print("=" * 60)
print("""
  Searching for specific features — even when described
  with completely different terminology.
""")

search_listings("smart home technology and modern appliances")
# Expected: Kakaako (motorized blinds, smart home) and Waikiki (premium finishes)

In [ ]:
search_listings("property with fruit trees and garden space")
# Expected: Big Island properties (coffee, macadamia, food forest)

### Demo 5: Why Keywords Fail

This demo shows what you would miss with traditional keyword search. We simulate a keyword search and then compare it to semantic search results.

In [ ]:
# ============================================================
# DEMO 5: Comparison with Keyword Approach
# ============================================================
# Show what you'd miss with traditional keyword search.

print("\n" + "=" * 60)
print("DEMO 5: Why Keywords Fail")
print("=" * 60)

query = "budget-friendly beachside home"

print(f'\n  Query: "{query}"\n')

# Simulate keyword search — check for exact word matches
keywords = query.lower().split()
print(f"  Keywords to match: {keywords}\n")

print("  --- Keyword Search Results ---")
keyword_matches = 0
for listing in PROPERTY_LISTINGS:
    text = (listing["title"] + " " + listing["description"]).lower()
    matching_keywords = [kw for kw in keywords if kw in text]
    if len(matching_keywords) >= 2:  # At least 2 keyword matches
        keyword_matches += 1
        print(f"    MATCH: {listing['title']} (matched: {matching_keywords})")

if keyword_matches == 0:
    print("    NO MATCHES FOUND! None of the listings contain these exact words.")

print(f"\n  Total keyword matches: {keyword_matches}")

# Now semantic search
print("\n  --- Semantic Search Results ---")
semantic_results = search_listings(query, n_results=3)
print(f"  Total semantic matches: {len(semantic_results)}")
print("  Semantic search found relevant listings by understanding MEANING!")

### Key Takeaways for Real Estate Professionals

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("KEY TAKEAWAYS FOR REAL ESTATE PROFESSIONALS:")
print("=" * 60)
print("""
  1. BUYERS SEARCH BY MEANING, NOT KEYWORDS
     "I want a place near the water that won't break the bank"
     doesn't match "oceanfront" or "affordable" — but semantic
     search understands the intent.

  2. EVERY AGENT WRITES DIFFERENTLY
     One says "panoramic views," another says "unobstructed vistas."
     Semantic search treats these as the same concept.

  3. LIFESTYLE MATCHING IS POWERFUL
     When a buyer says "I want the surfer lifestyle," semantic
     search finds the North Shore cottage — even without the
     word "surfer" in the listing.

  4. COMBINE WITH FILTERS FOR BEST RESULTS
     Use semantic search for the "feel" and structured filters
     for the hard requirements (beds, price, location).

  5. THIS IS THE FOUNDATION FOR RAG
     In Module 4, we'll connect this search capability to an
     AI chatbot that can answer questions about your listings,
     market reports, and business documents.
""")

### Clean Up

Delete the Pinecone index to avoid ongoing charges.

In [ ]:
# Clean up
pc.delete_index(INDEX_NAME)